# Go for ML Model Serving

**Objective:** Build a production-grade HTTP API in Go that serves ML model predictions — REST endpoints, JSON handling, middleware, and performance considerations.

**Prerequisites:** Concurrency patterns (notebook 02)

## 1. HTTP Server Basics (net/http)

In [ ]:
import (
    "fmt"
    "net/http"
)

// Basic handler function
http.HandleFunc("/hello", func(w http.ResponseWriter, r *http.Request) {
    fmt.Fprintf(w, "Hello, World!")
})

http.HandleFunc("/health", func(w http.ResponseWriter, r *http.Request) {
    w.Header().Set("Content-Type", "application/json")
    fmt.Fprintf(w, `{"status": "healthy"}`)
})

// Note: In a notebook, we can't run ListenAndServe (it blocks forever).
// In production, you'd call:
// http.ListenAndServe(":8080", nil)

fmt.Println("Handler functions registered for /hello and /health")
fmt.Println("In production: http.ListenAndServe(\":8080\", nil)")

## 2. JSON Encoding and Decoding

In [ ]:
import (
    "encoding/json"
    "fmt"
)

// Struct with JSON tags
type PredictionRequest struct {
    Features []float64 `json:"features"`
    ModelID  string    `json:"model_id"`
}

type PredictionResponse struct {
    Prediction float64 `json:"prediction"`
    Confidence float64 `json:"confidence"`
    ModelID    string  `json:"model_id"`
}

// Encoding (struct → JSON)
resp := PredictionResponse{
    Prediction: 0.85,
    Confidence: 0.92,
    ModelID:    "linear-v1",
}

jsonBytes, _ := json.MarshalIndent(resp, "", "  ")
fmt.Println("Encoded:")
fmt.Println(string(jsonBytes))

// Decoding (JSON → struct)
inputJSON := `{"features": [1.5, 2.3, 0.8], "model_id": "linear-v1"}`
var req PredictionRequest
json.Unmarshal([]byte(inputJSON), &req)
fmt.Printf("\nDecoded: %+v\n", req)
fmt.Printf("Features: %v, Model: %s\n", req.Features, req.ModelID)

## 3. Building a Prediction API

In [ ]:
import (
    "encoding/json"
    "fmt"
    "math"
    "net/http"
)

type PredictRequest struct {
    Features []float64 `json:"features"`
}

type PredictResponse struct {
    Prediction float64 `json:"prediction"`
    Class      string  `json:"class"`
}

// Simple sigmoid "model"
func sigmoid(x float64) float64 {
    return 1.0 / (1.0 + math.Exp(-x))
}

func predict(features []float64) (float64, string) {
    sum := 0.0
    weights := []float64{0.5, -0.3, 0.8}
    for i, f := range features {
        if i < len(weights) {
            sum += f * weights[i]
        }
    }
    prob := sigmoid(sum)
    class := "negative"
    if prob >= 0.5 {
        class = "positive"
    }
    return prob, class
}

// Demo the prediction function
testFeatures := []float64{1.0, -0.5, 2.0}
prob, class := predict(testFeatures)
fmt.Printf("Features: %v\n", testFeatures)
fmt.Printf("Prediction: %.4f (%s)\n", prob, class)

resp := PredictResponse{Prediction: prob, Class: class}
jsonBytes, _ := json.MarshalIndent(resp, "", "  ")
fmt.Printf("\nAPI Response:\n%s\n", string(jsonBytes))

## 4. Middleware (Logging, CORS, Auth)

In [ ]:
import (
    "fmt"
    "net/http"
    "time"
)

// Middleware pattern: wrap a handler
type Middleware func(http.HandlerFunc) http.HandlerFunc

func loggingMiddleware(next http.HandlerFunc) http.HandlerFunc {
    return func(w http.ResponseWriter, r *http.Request) {
        start := time.Now()
        next(w, r)
        duration := time.Since(start)
        fmt.Printf("[%s] %s %s — %v\n", r.Method, r.URL.Path, r.RemoteAddr, duration)
    }
}

func corsMiddleware(next http.HandlerFunc) http.HandlerFunc {
    return func(w http.ResponseWriter, r *http.Request) {
        w.Header().Set("Access-Control-Allow-Origin", "*")
        w.Header().Set("Access-Control-Allow-Methods", "GET, POST, OPTIONS")
        w.Header().Set("Access-Control-Allow-Headers", "Content-Type, Authorization")
        if r.Method == "OPTIONS" {
            w.WriteHeader(http.StatusOK)
            return
        }
        next(w, r)
    }
}

// Chain middleware
func chain(handler http.HandlerFunc, middlewares ...Middleware) http.HandlerFunc {
    for i := len(middlewares) - 1; i >= 0; i-- {
        handler = middlewares[i](handler)
    }
    return handler
}

fmt.Println("Middleware pattern:")
fmt.Println("  handler := chain(predictHandler, loggingMiddleware, corsMiddleware)")
fmt.Println("  http.HandleFunc(\"/predict\", handler)")

## 5. Loading and Running ONNX Models

In [ ]:
import "fmt"

fmt.Println("ONNX Model Serving in Go")
fmt.Println("========================")
fmt.Println("")
fmt.Println("Libraries:")
fmt.Println("  - github.com/owulveryck/onnx-go (pure Go ONNX runtime)")
fmt.Println("  - onnxruntime-go (CGo wrapper around Microsoft's ONNX Runtime)")
fmt.Println("")
fmt.Println("Workflow:")
fmt.Println("  1. Train model in Python (PyTorch/sklearn)")
fmt.Println("  2. Export to ONNX format: torch.onnx.export(model, ...)")
fmt.Println("  3. Load in Go:")
fmt.Println("     model, _ := onnx.NewModel(\"model.onnx\")")
fmt.Println("     input := tensor.New(tensor.WithShape(1, 10), tensor.Of(tensor.Float32))")
fmt.Println("     model.SetInput(0, input)")
fmt.Println("     _ = machine.Run(model)")
fmt.Println("     output, _ := model.GetOutputTensors()")
fmt.Println("")
fmt.Println("Advantage: Go's HTTP server + ONNX = fast, concurrent model serving")
fmt.Println("           without Python's GIL bottleneck")

## 6. Benchmarking and Performance

In [ ]:
import (
    "fmt"
    "math"
    "time"
)

// Benchmark the predict function
iterations := 100000
start := time.Now()

for i := 0; i < iterations; i++ {
    features := []float64{float64(i) * 0.001, -0.5, 2.0}
    sum := 0.0
    weights := []float64{0.5, -0.3, 0.8}
    for j, f := range features {
        if j < len(weights) {
            sum += f * weights[j]
        }
    }
    _ = 1.0 / (1.0 + math.Exp(-sum))
}

elapsed := time.Since(start)
fmt.Printf("Benchmark: %d predictions in %v\n", iterations, elapsed)
fmt.Printf("Throughput: %.0f predictions/sec\n", float64(iterations)/elapsed.Seconds())
fmt.Printf("Latency: %v per prediction\n", elapsed/time.Duration(iterations))
fmt.Println("")
fmt.Println("Go benchmark tools:")
fmt.Println("  go test -bench=. -benchmem")
fmt.Println("  hey -n 10000 -c 100 http://localhost:8080/predict")

## Try It Yourself

1. Build a REST API with `/predict` and `/health` endpoints. The predict endpoint should accept JSON input and return a mock prediction.
2. Add request logging middleware that records method, path, status code, and latency.
3. Benchmark your API with Go's `testing.B` or a tool like `hey`. How many requests per second can it handle?

In [ ]:
// Your code here